In [36]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import optuna
import joblib
from pathlib import Path

from optuna.samplers import TPESampler

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

In [37]:
BASE_DIR = Path.cwd()
TRAIN_PATH = BASE_DIR / "train_loan_final.csv"
TEST_PATH = BASE_DIR / "test_loan_final.csv"

df_train = pd.read_csv(TRAIN_PATH)
df_test = pd.read_csv(TEST_PATH)

In [38]:
df_train.head()
df_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 32670 entries, 0 to 32669
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   cust_id           32670 non-null  int64  
 1   income_yearly     31038 non-null  float64
 2   debt_ratio        32670 non-null  str    
 3   credit_rating     31038 non-null  float64
 4   amount_requested  32670 non-null  float64
 5   interest_pct      32670 non-null  float64
 6   gender_type       32670 non-null  str    
 7   family_status     32670 non-null  str    
 8   edu_bg            31038 non-null  str    
 9   job_status        31038 non-null  str    
 10  loan_reason       32670 non-null  str    
 11  internal_grade    32670 non-null  str    
 12  is_repaid         32670 non-null  float64
dtypes: float64(5), int64(1), str(7)
memory usage: 3.2 MB


In [39]:
df_train = df_train.drop(columns=["cust_id"])
cat_cols = ["gender_type", "family_status", "edu_bg", "job_status", "loan_reason", "internal_grade"]

for df in [df_train, df_test]:
    df["debt_ratio"] = pd.to_numeric(df["debt_ratio"], errors="coerce")

    for col in cat_cols:
        df[col] = df[col].astype(object).str.strip()

    df["gender_type"] = df["gender_type"].replace({"mAle": "Male"})
    df["job_status"] = df["job_status"].replace({"Employed_": "Employed"})

y = df_train["is_repaid"]
X = df_train.drop(columns=["is_repaid"])

X_test = df_test.drop(columns=["cust_id"])

df_train.info()
df_train.isna().sum()

<class 'pandas.DataFrame'>
RangeIndex: 32670 entries, 0 to 32669
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   income_yearly     31038 non-null  float64
 1   debt_ratio        32025 non-null  float64
 2   credit_rating     31038 non-null  float64
 3   amount_requested  32670 non-null  float64
 4   interest_pct      32670 non-null  float64
 5   gender_type       32670 non-null  object 
 6   family_status     32670 non-null  object 
 7   edu_bg            31038 non-null  object 
 8   job_status        31038 non-null  object 
 9   loan_reason       32670 non-null  object 
 10  internal_grade    32670 non-null  object 
 11  is_repaid         32670 non-null  float64
dtypes: float64(6), object(6)
memory usage: 3.0+ MB


income_yearly       1632
debt_ratio           645
credit_rating       1632
amount_requested       0
interest_pct           0
gender_type            0
family_status          0
edu_bg              1632
job_status          1632
loan_reason            0
internal_grade         0
is_repaid              0
dtype: int64

In [ ]:
#config
RANDOM_STATE = 42

In [ ]:
NUMERIC_COLUMNS = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
CATEGORICAL_COLUMNS = X.select_dtypes(include=['object', 'string']).columns.tolist()

numeric_transformer = SimpleImputer(strategy='median')

categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy="constant", fill_value="Missing")),
        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]
)

preprocessing = ColumnTransformer(
    transformers=[
        ('numeric', numeric_transformer, NUMERIC_COLUMNS),
        ('cat', categorical_transformer, CATEGORICAL_COLUMNS)
    ],
    remainder="drop"
)

In [7]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

skf = StratifiedKFold(
    n_splits = 5,
    shuffle = True,
    random_state=RANDOM_STATE
)

In [ ]:
def objective(trial):
    # XGBoost hyperparameters
    xgb_model = XGBClassifier(
        n_estimators=trial.suggest_int("xgb_n_estimators", 100, 800),
        max_depth=trial.suggest_int("xgb_max_depth", 2, 10),
        learning_rate=trial.suggest_float("xgb_learning_rate", 0.01, 0.3, log=True),
        subsample=trial.suggest_float("xgb_subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("xgb_colsample_bytree", 0.6, 1.0),
        min_child_weight=trial.suggest_int("xgb_min_child_weight", 1, 10),
        reg_alpha=trial.suggest_float("xgb_reg_alpha", 1e-8, 10.0, log=True),
        reg_lambda=trial.suggest_float("xgb_reg_lambda", 1e-8, 10.0, log=True),
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )


    # LightGBM hyperparameters
    lgbm_model = LGBMClassifier(
        n_estimators=trial.suggest_int("lgbm_n_estimators", 100, 600),
        max_depth=trial.suggest_int("lgbm_max_depth", 2, 12),
        learning_rate=trial.suggest_float("lgbm_learning_rate", 0.01, 0.3, log=True),
        num_leaves=trial.suggest_int("lgbm_num_leaves", 15, 127),
        subsample=trial.suggest_float("lgbm_subsample", 0.6, 1.0),
        colsample_bytree=trial.suggest_float("lgbm_colsample_bytree", 0.6, 1.0),
        reg_alpha=trial.suggest_float("lgbm_reg_alpha", 1e-8, 10.0, log=True),
        reg_lambda=trial.suggest_float("lgbm_reg_lambda", 1e-8, 10.0, log=True),
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1
    )

    # CatBoost hyperparameters
    cat_model = CatBoostClassifier(
        iterations=trial.suggest_int("cat_iterations", 100, 600),
        depth=trial.suggest_int("cat_depth", 2, 10),
        learning_rate=trial.suggest_float("cat_learning_rate", 0.01, 0.3, log=True),
        l2_leaf_reg=trial.suggest_float("cat_l2_leaf_reg", 1e-3, 10.0, log=True),
        random_seed=RANDOM_STATE,
        verbose=0,
        loss_function="Logloss"
    )

    # Meta model
    meta_model = LogisticRegression(
        C=trial.suggest_float("meta_C", 1e-4, 100.0, log=True),
        max_iter=3000,
        class_weight="balanced",
        random_state=RANDOM_STATE
    )

    # Stacking model
    stack_model = StackingClassifier(
        estimators=[
            ("xgb", xgb_model),
            ("lgbm", lgbm_model),
            ("cat", cat_model)
        ],
        final_estimator=meta_model,
        cv=skf,
        stack_method="predict_proba",
        n_jobs=-1,
        passthrough=False
    )

    # Full pipeline:
    # preprocessing -> stacking
    full_pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessing),
            ("model", stack_model)
        ]
    )

    scores = cross_val_score(
        full_pipeline,
        X_train,
        y_train,
        cv=skf,
        scoring="f1_macro",
        n_jobs=-1
    )

    return scores.mean()

In [9]:
study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=42)
)

study.optimize(
    objective,
    n_trials=50,
    show_progress_bar=True
)

print("Best F1-Macro:", study.best_value)
print("Best parameters:")
print(study.best_params)

[I 2026-05-14 12:04:05,582] A new study created in memory with name: no-name-9dc1e72f-446f-4cd7-be6a-06ea3dfd95fd
Best trial: 0. Best value: 0.82003:   2%|▏         | 1/50 [01:10<57:14, 70.10s/it]

[I 2026-05-14 12:05:15,683] Trial 0 finished with value: 0.8200300834561368 and parameters: {'xgb_n_estimators': 362, 'xgb_max_depth': 10, 'xgb_learning_rate': 0.1205712628744377, 'xgb_subsample': 0.8394633936788146, 'xgb_colsample_bytree': 0.6624074561769746, 'xgb_min_child_weight': 2, 'xgb_reg_alpha': 3.3323645788192616e-08, 'xgb_reg_lambda': 0.6245760287469893, 'lgbm_n_estimators': 401, 'lgbm_max_depth': 9, 'lgbm_learning_rate': 0.010725209743171996, 'lgbm_num_leaves': 124, 'lgbm_subsample': 0.9329770563201687, 'lgbm_colsample_bytree': 0.6849356442713105, 'lgbm_reg_alpha': 4.329370014459266e-07, 'lgbm_reg_lambda': 4.4734294104626844e-07, 'cat_iterations': 252, 'cat_depth': 6, 'cat_learning_rate': 0.04345454109729477, 'cat_l2_leaf_reg': 0.014618962793704957, 'meta_C': 0.4689400963537689}. Best is trial 0 with value: 0.8200300834561368.


Best trial: 0. Best value: 0.82003:   4%|▍         | 2/50 [01:46<40:18, 50.38s/it]

[I 2026-05-14 12:05:52,261] Trial 1 finished with value: 0.8108283836546679 and parameters: {'xgb_n_estimators': 197, 'xgb_max_depth': 4, 'xgb_learning_rate': 0.03476649150592621, 'xgb_subsample': 0.7824279936868144, 'xgb_colsample_bytree': 0.9140703845572055, 'xgb_min_child_weight': 2, 'xgb_reg_alpha': 0.00042472707398058225, 'xgb_reg_lambda': 0.0021465011216654484, 'lgbm_n_estimators': 123, 'lgbm_max_depth': 8, 'lgbm_learning_rate': 0.0178601378893971, 'lgbm_num_leaves': 22, 'lgbm_subsample': 0.9795542149013333, 'lgbm_colsample_bytree': 0.9862528132298237, 'lgbm_reg_alpha': 0.18861495878553936, 'lgbm_reg_lambda': 5.514725787121931e-06, 'cat_iterations': 148, 'cat_depth': 8, 'cat_learning_rate': 0.044684675025045834, 'cat_l2_leaf_reg': 0.003077180271250686, 'meta_C': 0.09355380606452186}. Best is trial 0 with value: 0.8200300834561368.


Best trial: 2. Best value: 0.821305:   6%|▌         | 3/50 [03:24<56:18, 71.88s/it]

...

[I 2026-05-14 13:04:39,583] Trial 47 finished with value: 0.8314488343702596 and parameters: {'xgb_n_estimators': 251, 'xgb_max_depth': 10, 'xgb_learning_rate': 0.0161021231078746, 'xgb_subsample': 0.9127091048361118, 'xgb_colsample_bytree': 0.8134146320379265, 'xgb_min_child_weight': 7, 'xgb_reg_alpha': 1.2186817597191351e-08, 'xgb_reg_lambda': 0.001468684913974319, 'lgbm_n_estimators': 575, 'lgbm_max_depth': 12, 'lgbm_learning_rate': 0.1354826023470676, 'lgbm_num_leaves': 77, 'lgbm_subsample': 0.7500249311915983, 'lgbm_colsample_bytree': 0.9784678102135531, 'lgbm_reg_alpha': 1.6393672751882377e-06, 'lgbm_reg_lambda': 6.6245420693767185, 'cat_iterations': 148, 'cat_depth': 4, 'cat_learning_rate': 0.06882478250782267, 'cat_l2_leaf_reg': 0.0027726129527086353, 'meta_C': 0.00024243716056967528}. Best is trial 44 with value: 0.8380067095175316.


Best trial: 44. Best value: 0.838007:  98%|█████████▊| 49/50 [1:01:54<01:13, 73.46s/it]

[I 2026-05-14 13:05:59,750] Trial 48 finished with value: 0.8333410533850032 and parameters: {'xgb_n_estimators': 146, 'xgb_max_depth': 10, 'xgb_learning_rate': 0.018772688030325536, 'xgb_subsample': 0.9304932594386797, 'xgb_colsample_bytree': 0.9201969056852639, 'xgb_min_child_weight': 8, 'xgb_reg_alpha': 6.602469182393772e-08, 'xgb_reg_lambda': 0.004871771569257112, 'lgbm_n_estimators': 593, 'lgbm_max_depth': 10, 'lgbm_learning_rate': 0.08383528711284347, 'lgbm_num_leaves': 83, 'lgbm_subsample': 0.8641523320372558, 'lgbm_colsample_bytree': 0.9232252942752782, 'lgbm_reg_alpha': 1.1464905351690616e-08, 'lgbm_reg_lambda': 0.6278634482712748, 'cat_iterations': 124, 'cat_depth': 3, 'cat_learning_rate': 0.03381183150575458, 'cat_l2_leaf_reg': 0.00418046574747948, 'meta_C': 0.00011489212378165262}. Best is trial 44 with value: 0.8380067095175316.


Best trial: 44. Best value: 0.838007: 100%|██████████| 50/50 [1:03:23<00:00, 76.07s/it]

[I 2026-05-14 13:07:29,243] Trial 49 finished with value: 0.8355387745815666 and parameters: {'xgb_n_estimators': 278, 'xgb_max_depth': 10, 'xgb_learning_rate': 0.015407484047874785, 'xgb_subsample': 0.619358102474579, 'xgb_colsample_bytree': 0.8750901080122574, 'xgb_min_child_weight': 1, 'xgb_reg_alpha': 3.3299884641953694e-08, 'xgb_reg_lambda': 0.000616203257238676, 'lgbm_n_estimators': 497, 'lgbm_max_depth': 11, 'lgbm_learning_rate': 0.15005785166320154, 'lgbm_num_leaves': 71, 'lgbm_subsample': 0.727186891702704, 'lgbm_colsample_bytree': 0.9546132918131258, 'lgbm_reg_alpha': 6.025807537109263e-08, 'lgbm_reg_lambda': 0.1215129012673256, 'cat_iterations': 132, 'cat_depth': 4, 'cat_learning_rate': 0.1489709417405859, 'cat_l2_leaf_reg': 0.010657549079951863, 'meta_C': 0.0012008671138771779}. Best is trial 44 with value: 0.8380067095175316.
Best F1-Macro: 0.8380067095175316
Best parameters:
{'xgb_n_estimators': 104, 'xgb_max_depth': 10, 'xgb_learning_rate': 0.015018727307270193, 'xgb_sub

In [12]:
# Compare base models using Optuna best parameters
best_params = study.best_params

base_models = {
    "xgb": XGBClassifier(
        n_estimators=best_params["xgb_n_estimators"],
        max_depth=best_params["xgb_max_depth"],
        learning_rate=best_params["xgb_learning_rate"],
        subsample=best_params["xgb_subsample"],
        colsample_bytree=best_params["xgb_colsample_bytree"],
        min_child_weight=best_params["xgb_min_child_weight"],
        reg_alpha=best_params["xgb_reg_alpha"],
        reg_lambda=best_params["xgb_reg_lambda"],
        objective="binary:logistic",
        eval_metric="logloss",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ),
    "lgbm": LGBMClassifier(
        n_estimators=best_params["lgbm_n_estimators"],
        max_depth=best_params["lgbm_max_depth"],
        learning_rate=best_params["lgbm_learning_rate"],
        num_leaves=best_params["lgbm_num_leaves"],
        subsample=best_params["lgbm_subsample"],
        colsample_bytree=best_params["lgbm_colsample_bytree"],
        reg_alpha=best_params["lgbm_reg_alpha"],
        reg_lambda=best_params["lgbm_reg_lambda"],
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1
    ),
    "cat": CatBoostClassifier(
        iterations=best_params["cat_iterations"],
        depth=best_params["cat_depth"],
        learning_rate=best_params["cat_learning_rate"],
        l2_leaf_reg=best_params["cat_l2_leaf_reg"],
        random_seed=RANDOM_STATE,
        verbose=0,
        loss_function="Logloss"
    )
}

base_model_results = []
base_model_pipelines = {}

for name, model in base_models.items():
    pipe = Pipeline(
        steps=[
            ("preprocessor", preprocessing),
            ("model", model)
        ]
    )

    pipe.fit(X_train, y_train)
    base_model_pipelines[name] = pipe

    y_pred = pipe.predict(X_valid)
    y_proba = pipe.predict_proba(X_valid)[:, 1]

    best_threshold = 0.5
    best_macro_f1 = f1_score(y_valid, y_pred, average="macro")

    for threshold in np.linspace(0.05, 0.95, 181):
        threshold_pred = (y_proba >= threshold).astype(int)
        threshold_macro_f1 = f1_score(y_valid, threshold_pred, average="macro")

        if threshold_macro_f1 > best_macro_f1:
            best_threshold = threshold
            best_macro_f1 = threshold_macro_f1

    base_model_results.append({
        "model": name,
        "accuracy_default_0_5": accuracy_score(y_valid, y_pred),
        "macro_f1_default_0_5": f1_score(y_valid, y_pred, average="macro"),
        "roc_auc": roc_auc_score(y_valid, y_proba),
        "best_threshold": best_threshold,
        "macro_f1_best_threshold": best_macro_f1
    })

base_model_results = pd.DataFrame(base_model_results).sort_values(
    "macro_f1_best_threshold",
    ascending=False
)

base_model_results


,model,accuracy_default_0_5,macro_f1_default_0_5,roc_auc,best_threshold,macro_f1_best_threshold
1,lgbm,0.911693,0.849967,0.913943,0.560,0.852575
0,xgb,0.904346,0.821250,0.909216,0.685,0.837525
2,cat,0.904806,0.828650,0.911605,0.670,0.834683


In [13]:
best_params = study.best_params

final_xgb = XGBClassifier(
    n_estimators=best_params["xgb_n_estimators"],
    max_depth=best_params["xgb_max_depth"],
    learning_rate=best_params["xgb_learning_rate"],
    subsample=best_params["xgb_subsample"],
    colsample_bytree=best_params["xgb_colsample_bytree"],
    min_child_weight=best_params["xgb_min_child_weight"],
    reg_alpha=best_params["xgb_reg_alpha"],
    reg_lambda=best_params["xgb_reg_lambda"],
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

final_lgbm = LGBMClassifier(
    n_estimators=best_params["lgbm_n_estimators"],
    max_depth=best_params["lgbm_max_depth"],
    learning_rate=best_params["lgbm_learning_rate"],
    num_leaves=best_params["lgbm_num_leaves"],
    subsample=best_params["lgbm_subsample"],
    colsample_bytree=best_params["lgbm_colsample_bytree"],
    reg_alpha=best_params["lgbm_reg_alpha"],
    reg_lambda=best_params["lgbm_reg_lambda"],
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1
)

final_cat = CatBoostClassifier(
    iterations=best_params["cat_iterations"],
    depth=best_params["cat_depth"],
    learning_rate=best_params["cat_learning_rate"],
    l2_leaf_reg=best_params["cat_l2_leaf_reg"],
    random_seed=RANDOM_STATE,
    verbose=0,
    loss_function="Logloss"
)

final_meta = LogisticRegression(
    C=best_params["meta_C"],
    max_iter=3000,
    class_weight="balanced",
    random_state=RANDOM_STATE
)

final_stack_model = StackingClassifier(
    estimators=[
        ("xgb", final_xgb),
        ("lgbm", final_lgbm),
        ("cat", final_cat)
    ],
    final_estimator=final_meta,
    cv=skf,
    stack_method="predict_proba",
    n_jobs=-1,
    passthrough=False
)

final_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessing),
        ("model", final_stack_model)
    ]
)

final_pipeline.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numeric', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformer

In [29]:
test_pred = final_pipeline.predict(df_test)
test_proba = final_pipeline.predict_proba(df_test)[:, 1]

test_predictions = pd.DataFrame({
    "id": df_test["cust_id"],
    "is_repaid": test_pred.astype(int)
})

test_predictions.to_csv("submission_loan_final.csv", index=False)
test_predictions.head()


,id,is_repaid
0,1,1
1,2,1
2,3,1
3,4,1
4,5,1
